<a href="https://colab.research.google.com/github/YashodhanSonune/SLMFromScratch/blob/main/SLMfromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP-1: Loading Dataset

In [1]:
print("SLM FROM SCRATCH")

SLM FROM SCRATCH


In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset
ds = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

STEP-2: Tokenization

In [4]:
#Using tiktoken library for Byte Pair Encoding (used in GPT-2)

!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

In [5]:
enc = tiktoken.get_encoding("gpt2")

def process(example):
  ids = enc.encode_ordinary(example['text']) # encode_ordinary ignores special tokens
  out = {'ids': ids, 'len': len(ids)}
  return out

if not os.path.exists("train.bin"):
  tokenized = ds.map(
      process,
      remove_columns = ['text'],
      desc = "tokenizing the splits",
      num_proc = 8
  )

  # concatenate all the ids in each dataset into one large file we can use for training

  for split, dset in tokenized.items():
    arr_len = np.sum(dset['len'], dtype = np.uint64)
    filename = f'{split}.bin'
    dtype = np.uint16
    arr = np.memmap(filename, dtype = dtype, mode = 'w+', shape = (arr_len))
    total_batches = 1024

    idx = 0
    for batch_idx in tqdm(range(total_batches), desc = f'writing {filename}'):
      # Batch together samples for faster write
      batch = dset.shard(num_shards = total_batches, index = batch_idx, contiguous = True).with_format('numpy')
      arr_batch = np.concatenate(batch['ids'])

      # Write into mmap
      arr[idx : idx + len(arr_batch)] = arr_batch
      idx += len(arr_batch)

    arr.flush()


tokenizing the splits (num_proc=8):   0%|          | 0/2119719 [00:00<?, ? examples/s]

tokenizing the splits (num_proc=8):   0%|          | 0/21990 [00:00<?, ? examples/s]

writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

Step-3: Create Input-Output batches for the Dataset

In [1]:
# Block size = Context window
def get_batch(split):
  if split == 'train':
    data = np.memmap('train.bin', dtype = np.uint16, mode = 'r')
  else:
    data = np.memmap('validation.bin', dtype = np.uint16, mode = 'r')

  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([torch.from_numpy((data[i:i + batch_size]).astype(np.int64)) for i in ix])
  y = torch.stack([torch.from_numpy((data[i + 1:i + 1 + batch_size]).astype(np.int64)) for i in ix])

  if device_type == "cuda":
    # pin arrays x, y which allows us to move them to GPU asynchronously (non_blocking = True)
    x, y = x.pin_memory().to(device, non_blocking = True), y.pin_memory().to(device, non_blocking = True)
  else:
    x, y = x.to(device), y.to(device)
  return x, y
